# Baseline modellek — Mean, Median, Lineáris regresszió

Ez a notebook **összehasonlítási referenciapontokat** tanít: két naive baseline-t (mean, median) és egy Ridge regularizált lineáris regressziót. A három modell ugyanazt a feature-szettet kapja, mint a Decision Tree (`03_decision_tree.ipynb`), így az eredményeik közvetlenül összehasonlíthatók.

**Miért kell ez:**
- A Decision Tree MAE önmagában értelmezhetetlen — a median-baseline-hoz és a lineáris regresszióhoz viszonyítva tudjuk megmondani, mennyit ad hozzá a fa nemlinearitása.
- A `04_comparison.ipynb` automatikusan beolvassa az itt írt sorokat a `metrics.csv`-ből.

Mentett kimenetek (Drive: `MODELS_DIR`):
- `linear_ridge.joblib` — a Ridge regresszor
- `linear_predictions_test.npy` — a Ridge teszt-predikciói (későbbi reziduum-elemzéshez)

Plus `results/metrics.csv` (Drive) — három append-elt sor: `Baseline (mean)`, `Baseline (median)`, `Linear Regression (Ridge)`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/02_baselines.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'main'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone --branch {BRANCH} https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')
print(f'Branch: {BRANCH}')

In [ ]:
import numpy as np
import joblib
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge

from src.config import RANDOM_SEED, MODELS_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.evaluation import append_metrics, evaluate_model

ensure_dirs()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Adatbetöltés

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_v1_data()

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Target — train: mean={y_train.mean():.2f}, median={np.median(y_train):.2f}, '
      f'std={y_train.std():.2f}, min={y_train.min()}, max={y_train.max()}')

## 3. Mean baseline

Mindig `y_train.mean()`-t jósolja, függetlenül a feature-öktől. Ez az **MSE-optimális konstans prediktor** — alacsonyabb RMSE-t (más konstanssal) nem lehet elérni. R² = 0 definíció szerint a *train* átlagra; a teszten kicsit eltérhet, ha a test mean ≠ train mean.

In [ ]:
mean_baseline = DummyRegressor(strategy='mean')
result_mean, _ = evaluate_model(
    mean_baseline,
    X_train, y_train, X_test, y_test,
    model_name='Baseline (mean)',
    notes='naive: mindig y_train átlagát jósolja',
)
append_metrics(result_mean)
print(f'MAE: {result_mean.mae:.3f}')
print(f'RMSE: {result_mean.rmse:.3f}')
print(f'R²: {result_mean.r2:.3f}')

## 4. Median baseline

Mindig `np.median(y_train)`-t jósolja. Ez az **MAE-optimális konstans prediktor** — alacsonyabb MAE-t (más konstanssal) nem lehet elérni. Mivel a target jobbra ferde (mean > median), a median-baseline tipikusan jobb MAE-t ad, mint a mean-baseline.

In [ ]:
median_baseline = DummyRegressor(strategy='median')
result_median, _ = evaluate_model(
    median_baseline,
    X_train, y_train, X_test, y_test,
    model_name='Baseline (median)',
    notes='naive: mindig y_train medianját jósolja',
)
append_metrics(result_median)
print(f'MAE: {result_median.mae:.3f}')
print(f'RMSE: {result_median.rmse:.3f}')
print(f'R²: {result_median.r2:.3f}')

## 5. Lineáris regresszió (Ridge)

Ugyanazon a feature-szetten, mint a Decision Tree. **Ridge** (L2-regulárizált lineáris regresszió) a plain `LinearRegression` helyett, mert:
- A one-hot oszlopok kollineárisak lehetnek
- A `cyclic encoding` sin/cos párjai is korreláltak
- `alpha=1.0` numerikusan stabil, nem hangoljuk (a baseline lényege a *nem hangolt* viszonyítás)

A Ridge megmutatja, mit ad hozzá önmagában a feature-szet, **mielőtt** bármi nem-lineáris struktúrát modelleznénk. A Decision Tree és a Ridge MAE-jének különbsége = a fa által felfedezett nem-lineáris struktúra értéke.

In [ ]:
linear_model = Ridge(alpha=1.0, random_state=RANDOM_SEED)
result_linear, y_pred_linear = evaluate_model(
    linear_model,
    X_train, y_train, X_test, y_test,
    model_name='Linear Regression (Ridge)',
    notes=f'Ridge(alpha=1.0) on {X_train.shape[1]} features',
)
append_metrics(result_linear)
print(f'MAE: {result_linear.mae:.3f}')
print(f'RMSE: {result_linear.rmse:.3f}')
print(f'R²: {result_linear.r2:.3f}')
print(f'Train idő: {result_linear.train_time_sec:.2f} s')

joblib.dump(linear_model, MODELS_DIR / 'linear_ridge.joblib')
np.save(MODELS_DIR / 'linear_predictions_test.npy', y_pred_linear)
print(f'\nMentve: {MODELS_DIR}/linear_ridge.joblib + linear_predictions_test.npy')

## 6. Összegzés

Ez csak egy gyors helyi áttekintés — a teljes ranking és vizualizáció a `04_comparison.ipynb`-ben van, ami minden modell metrikáját beolvassa a CSV-ből.

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {'model': result_mean.model_name,   'MAE': result_mean.mae,   'RMSE': result_mean.rmse,   'R²': result_mean.r2},
    {'model': result_median.model_name, 'MAE': result_median.mae, 'RMSE': result_median.rmse, 'R²': result_median.r2},
    {'model': result_linear.model_name, 'MAE': result_linear.mae, 'RMSE': result_linear.rmse, 'R²': result_linear.r2},
]).round(3)
summary